# 32 | Mask R-CNN: repeated training with seeds 2027 and 2028

**Compute:** GPU T4 or better (Google Colab); **do not use CPU for training**. The original seed-2026 run from Experiment 06 is preserved unchanged.

**Scientific goal:** complement existing YOLO11n-seg and YOLO26n-seg three-seed training evidence with two additional Mask R-CNN ResNet50-FPN runs. Use the same FaroPigSeg date-disjoint train/validation partition and the **original Experiment 06 training recipe**. There is no test inference, no threshold tuning on test, and no modification of original model files.

**Training configurations:** COCO-pretrained weights, `min_size=512`, `max_size=768`, batch 1, AdamW LR=1e-4, weight decay 1e-4, up to 12 epochs, validation COCO mask AP every two epochs, early stopping after three validation checks without improvement, automatic mixed precision.

**Artifacts:** independent `seed_2027` and `seed_2028` folders in `03_Resultados/06_Entrenamiento/MaskRCNN_multisemilla_v1/`, plus Excel, English publication figures (600 DPI PNG, PDF, SVG), JSON protocols, and a ZIP without the large model weights (weights stay on Drive).

**Important:** these training-validation outcomes are not the final test comparison. YOLO and Mask R-CNN retain unequal training budgets. Any training stopped at epoch 12 without stabilized validation performance must be transparently reported.


## 1. Runtime and required packages

In Google Colab choose **Runtime → Change runtime type → GPU** (T4 or L4). Run all cells from the top. Do not reinstall PyTorch or torchvision from a generic wheel; use the compatible Colab GPU packages and record their actual versions.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. In Colab: Runtime > Change runtime type > GPU (T4 or better).')
print('GPU:', torch.cuda.get_device_name(0))
%pip -q install "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" "openpyxl>=3.1,<4" "pillow>=10" "opencv-python-headless>=4.8" "tqdm>=4.66" "matplotlib>=3.8"


## 2. Mount Drive and verify the frozen experimental split

This is a strict, fail-fast integrity check: every partition file is fingerprinted and the set sizes/dates are checked against the already audited Experiment 31. Validation is used only to select best weights; **test is never evaluated in this notebook**.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, random, gc, time, platform, sys, zipfile
import numpy as np, pandas as pd, cv2, yaml, matplotlib.pyplot as plt
import torch, torchvision
from PIL import Image

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Project folder unavailable: {ROOT}'
DATA = ROOT/'01_Datasets'/'FaroPigSeg'
SPLIT = DATA/'split_fecha_v1'
RES = ROOT/'03_Resultados'
EXPORT = ROOT/'05_Exportaciones_ZIP'
BASELINE = RES/'06_Entrenamiento'/'MaskRCNN_fecha_v1'
RUNS = RES/'06_Entrenamiento'/'MaskRCNN_multisemilla_v1'
REP = RES/'05_Reportes'/'CERDOS_32_MaskRCNN_Semillas'
FIG = RES/'04_Figuras'/'CERDOS_32_MaskRCNN_Semillas'
for path in (RUNS, REP, FIG, EXPORT):path.mkdir(parents=True, exist_ok=True)
YAML = SPLIT/'data_fecha.yaml'
MANIFEST = SPLIT/'particion_fecha_manifest.json'
EXPECTED_SHA = {
    'data_fecha.yaml':'9a223b47038c661303c1d12a35cbdcb212ed51529c8c773c4c9c66998706d1d9',
    'particion_fecha_manifest.json':'27ba2e77c188d360bc5d6cbac2a2e2bf0ec929448ae082e49e583e222ad48cd3',
    'train.txt':'17b343f693a62714ebe0d1629b30ec8daff9c7081a1f4234f90ab11107921281',
    'val.txt':'6368e3c82d001e33035cdb46ea6c7882ad599d22a06d885a22d8177549619e9d',
    'test.txt':'0d31f2b3bc931c6f08c96544df8fdb1f91c3b6bea5da4624917e94da97e01c05'
}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
for name,expect in EXPECTED_SHA.items():
    actual=sha256_file(SPLIT/name)
    if actual!=expect:
        raise RuntimeError(f'Frozen split hash mismatch: {name}. Expected {expect}, got {actual}. Do not start training.')
partition=json.loads(MANIFEST.read_text(encoding='utf-8'))
config=yaml.safe_load(YAML.read_text(encoding='utf-8'))
assert all(k in config for k in ('train','val','test')), 'Data YAML must contain train/val/test'
files_by_split={}
for key,n,nd in [('train',949,16),('val',196,4),('test',373,5)]:
    file=SPLIT/f'{key}.txt'
    paths=[Path(s.strip()) for s in file.read_text(encoding='utf-8').splitlines() if s.strip()]
    assert len(paths)==n and len(set(map(str,paths)))==n, f'Unexpected {key} size or duplicates'
    assert Path(config[key]).resolve()==file.resolve(), f'YAML {key} does not point to frozen {key}.txt'
    dates={p.name[:10] for p in paths}
    assert len(dates)==nd, f'Unexpected dates in {key}: {len(dates)}'
    assert all(partition['asignaciones_fecha'].get(p.name[:10])==key for p in paths), f'Manifest mismatch in {key}'
    files_by_split[key]=paths
assert not(set(map(str, files_by_split['train']))&set(map(str,files_by_split['val'])))
assert not(set(map(str, files_by_split['val']))&set(map(str,files_by_split['test'])))
assert (BASELINE/'maskrcnn_best_model.pt').is_file(), 'Original best Mask R-CNN weights are missing. Stop.'
assert (BASELINE/'historial_maskrcnn.csv').is_file(), 'Original Mask R-CNN validation history missing.'
print('Verified partitions: train=949 (16 dates), validation=196 (4 dates), test=373 (5 dates).')
print('Baseline Mask R-CNN file retained:',BASELINE/'maskrcnn_best_model.pt')
print('Environment:',platform.python_version(), 'torch',torch.__version__, 'torchvision',torchvision.__version__)
print('No test inference is performed.')


## 3. Dataset, architecture and COCO evaluator — same implementation as Experiment 06

The following code is intentionally retained from Notebook 06 to keep masks, ground truth, transforms, architecture settings and validation scoring consistent. Individual pigs are converted from normalized label polygons to binary instance masks.


In [ ]:
import numpy as np, pandas as pd,torch,cv2,yaml,random,math
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torchvision.transforms.functional import to_tensor

class PigMasks(Dataset):
    def __init__(self,imagelist):
        self.paths=[Path(s.strip()) for s in Path(imagelist).read_text(encoding='utf8').splitlines() if s.strip()]
        if not self.paths:raise RuntimeError('La partición está vacía: '+str(imagelist))
        missing=[str(p) for p in self.paths if not p.is_file()]
        if missing: raise FileNotFoundError('Hay imágenes ausentes: '+str(missing[:3]))
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        p=self.paths[index]
        rgb=np.asarray(Image.open(p).convert('RGB'))
        h,w=rgb.shape[:2]
        lab=Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')
        if not lab.is_file():raise FileNotFoundError('Etiqueta no localizada: '+str(lab))
        masks=[];boxes=[]
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip(): continue
            vals=row.split()
            if int(vals[0])!=0:raise ValueError('Clase inesperada en '+str(lab))
            points=np.asarray([float(v) for v in vals[1:]],dtype=np.float32)
            if len(points)<6 or len(points)%2:raise ValueError('Polígono inválido en '+str(lab))
            if not np.isfinite(points).all() or np.any((points<0)|(points>1)):
                raise ValueError('Coordenadas no normalizadas en '+str(lab))
            coords=points.reshape(-1,2)*np.asarray([w,h],dtype=np.float32)
            xy=np.rint(coords).astype(np.int32)
            xy[:,0]=np.clip(xy[:,0],0,w-1);xy[:,1]=np.clip(xy[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[xy],1)
            yy,xx=np.where(mask)
            if len(xx)<3: continue
            x0,x1=int(xx.min()),int(xx.max()+1);y0,y1=int(yy.min()),int(yy.max()+1)
            if x1<=x0 or y1<=y0:continue
            masks.append(mask);boxes.append([x0,y0,x1,y1])
        if not masks:raise RuntimeError('Imagen sin máscaras válidas: '+p.name)
        tar={'boxes':torch.as_tensor(boxes,dtype=torch.float32),
             'labels':torch.ones((len(masks),),dtype=torch.int64),
             'masks':torch.as_tensor(np.stack(masks),dtype=torch.uint8),
             'image_id':torch.tensor([index],dtype=torch.int64),
             'area':torch.as_tensor([int(m.sum()) for m in masks],dtype=torch.float32),
             'iscrowd':torch.zeros((len(masks),),dtype=torch.int64)}
        return to_tensor(rgb.copy()),tar

def collate_fn(batch):return tuple(zip(*batch))

import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_maskrcnn(pretrained=True):
    model=maskrcnn_resnet50_fpn(weights='DEFAULT' if pretrained else None,
                               weights_backbone=None,min_size=512,max_size=768,
                               trainable_backbone_layers=3)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,2)
    in_channels=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(in_channels,256,2)
    # Congelar los mismos niveles, tanto en arranque con pesos COCO como en reanudación sin descarga.
    for name,parameter in model.backbone.body.named_parameters():
        if name.startswith(('conv1.', 'bn1.', 'layer1.')):
            parameter.requires_grad_(False)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def config_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils
from contextlib import redirect_stdout
import io

def encode_mask(arr):
    enc=maskutils.encode(np.asfortranarray(np.asarray(arr,dtype=np.uint8)))
    enc['counts']=enc['counts'].decode('ascii')
    return enc

def coco_metric_for_model(model,dataset,device,desc='Validación',include_rows=False):
    model.eval();gts=[];preds=[];image_records=[];annotations=[];aid=1
    for i in range(len(dataset)):
        image,target=dataset[i]
        height,width=image.shape[1:]
        gts.append({'id':i+1,'width':width,'height':height,'file_name':dataset.paths[i].name})
        gt_masks=target['masks'].numpy()
        for m in gt_masks:
            encoded=encode_mask(m)
            x,y,w,h=[float(v) for v in maskutils.toBbox({'size':encoded['size'],'counts':encoded['counts'].encode('ascii')})]
            annotations.append({'id':aid,'image_id':i+1,'category_id':1,'segmentation':encoded,
                                'bbox':[x,y,w,h],'area':int(m.sum()),'iscrowd':0});aid+=1
        with torch.no_grad():
            out=model([image.to(device)])[0]
        scores=out['scores'].detach().cpu().numpy()
        ms=(out['masks'].detach().cpu().numpy()[:,0]>.5) if len(scores) else np.zeros((0,height,width),dtype=bool)
        for j,score in enumerate(scores):
            if score<0.001: continue
            m=ms[j].astype(np.uint8)
            if not m.any():continue
            enc=encode_mask(m)
            preds.append({'image_id':i+1,'category_id':1,'segmentation':enc,'score':float(score)})
        if include_rows:
            filtered=np.where(scores>=.25)[0]
            predm=[ms[k].astype(bool) for k in filtered if k<len(ms)]
            p_scores=[float(scores[k]) for k in filtered if k<len(ms)]
            pairs=[]
            for gi,g in enumerate(gt_masks):
                for pj,pm in enumerate(predm):
                    inter=np.logical_and(g,pm).sum();union=np.logical_or(g,pm).sum()
                    iou=inter/max(1,union)
                    if iou>=.5:pairs.append((float(iou),gi,pj))
            used_gt=set();used_pred=set()
            for _,gi,pj in sorted(pairs,reverse=True):
                if gi not in used_gt and pj not in used_pred:
                    used_gt.add(gi);used_pred.add(pj)
            tp=len(used_gt)
            image_records.append({'imagen':dataset.paths[i].name,
                'fecha':dataset.paths[i].name[:10],
                'anotaciones_GT':len(gt_masks),'detecciones_conf_025':len(predm),
                'TP_IoU05':tp,'FP_IoU05':len(predm)-tp,'FN_IoU05':len(gt_masks)-tp,
                'confianza_media_pred':np.mean(p_scores) if p_scores else np.nan})
        if (i+1)%50==0:print(desc,i+1,'/',len(dataset))
    gt=COCO()
    gt.dataset={'images':gts,'annotations':annotations,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
        dt=gt.loadRes(preds) if preds else gt.loadRes([])
        ev=COCOeval(gt,dt,'segm');ev.evaluate();ev.accumulate();ev.summarize()
    return {'seg_map50_95':float(ev.stats[0]),'seg_map50':float(ev.stats[1]),
            'seg_map75':float(ev.stats[2]),'gt':len(annotations),
            'num_images':len(dataset),'pred_coarse':len(preds)},pd.DataFrame(image_records)


## 4. Train the two missing seeds and save independent checkpoints

Runs are sequential and resumable. If Colab disconnects, rerun all cells: completed seeds are skipped and an unfinished seed resumes from its last saved checkpoint. Each new run starts from COCO-pretrained weights with its own seed, not from the previously trained 2026 checkpoint. A successful run saves original-score validation performance, checkpoints, a reproducibility JSON and a completion marker. Full weights are not placed in the ZIP.


In [ ]:
import pandas as pd
SEEDS = (2027, 2028)
EPOCHS, BATCH, VAL_EVERY, PATIENCE_EVALS = 12, 1, 2, 3
LR, WEIGHT_DECAY = 0.0001, 0.0001
YAML_SHA = sha256_file(YAML)
DEVICE = torch.device('cuda:0')
train_ds = PigMasks(config['train'])
val_ds = PigMasks(config['val'])
assert len(train_ds)==949 and len(val_ds)==196
_, first_target=train_ds[0]
assert len(first_target['masks'])>0, 'Training dataset contains no masks in its first image'
print('Train:',len(train_ds),'Validation:',len(val_ds),'Example GT pigs:',len(first_target['masks']))

def initialize_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False

def atomic_save(obj,path):
    tmp=path.with_name(path.name+'.tmp')
    torch.save(obj,tmp)
    os.replace(tmp,path)

def cpu_model_state(model):
    return {k:v.detach().cpu() for k,v in model.state_dict().items()}

def train_seed(seed):
    initialize_seed(seed)
    outdir=RUNS/f'seed_{seed}'
    outdir.mkdir(parents=True,exist_ok=True)
    best=outdir/'maskrcnn_best_model.pt'
    ckpt=outdir/'maskrcnn_last_checkpoint.pt'
    done=outdir/'CERDOS_32_ENTRENAMIENTO_FINALIZADO.json'
    hist=outdir/'historial_maskrcnn.csv'
    if done.is_file():
        d=json.loads(done.read_text(encoding='utf-8'))
        if (d.get('seed')==seed and d.get('yaml_sha256')==YAML_SHA and
            best.is_file() and hist.is_file() and
            sha256_file(best)==d.get('best_weights_sha256')):
            print(f'Seed {seed}: verified completion marker, skipping training.')
            return d
        raise RuntimeError(f'Seed {seed}: completion marker or weights invalid: {outdir}. Inspect before continuing.')

    # COCO pretraining follows EXACT model construction in Experiment 06.
    # For resume, do not download a pretrained model unnecessarily.
    model=build_maskrcnn(pretrained=not ckpt.exists()).to(DEVICE)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,collate_fn=collate_fn)
    start_epoch=1
    best_ap=-1.0
    no_improve=0
    rows=[]
    if ckpt.is_file():
        state=torch.load(ckpt,map_location='cpu',weights_only=False)
        assert state['yaml_sha256']==YAML_SHA and state['seed']==seed, 'Checkpoint incompatible with current seed/split.'
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        start_epoch=int(state['epoch'])+1
        best_ap=float(state['best_ap']);no_improve=int(state['no_improve'])
        if 'rng_torch' in state:torch.set_rng_state(state['rng_torch'])
        if 'rng_cuda' in state:torch.cuda.set_rng_state_all(state['rng_cuda'])
        if 'rng_numpy' in state:np.random.set_state(state['rng_numpy'])
        if 'rng_python' in state:random.setstate(state['rng_python'])
        print(f'Seed {seed}: resuming at epoch {start_epoch}.')
        assert hist.is_file(), 'Checkpoint exists without epoch history; inspect before resuming.'
        rows=pd.read_csv(hist).to_dict('records')
        assert rows and len(rows)==start_epoch-1, 'Checkpoint/history epoch mismatch.'
        assert best.is_file(), 'Checkpoint exists but selected best weights are missing.'
    interrupted_at=None
    # A completed checkpoint at the final epoch may need its report finalized after a previous Colab disconnection.
    for epoch in range(start_epoch,EPOCHS+1):
        if no_improve>=PATIENCE_EVALS:
            print('Prior checkpoint already satisfied early stopping; finalizing.')
            break
        model.train();losses=[];t0=time.time()
        for images,targets in loader:
            images=[x.to(DEVICE) for x in images]
            targets=[{k:v.to(DEVICE) for k,v in tar.items()} for tar in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=True):
                dloss=model(images,targets)
                loss=sum(dloss.values())
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss at seed={seed}, epoch={epoch}')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.)
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.detach().cpu()))
        entry={'epoch':epoch,'train_loss_mean':float(np.mean(losses)),
               'seg_map50_95_val':np.nan,'seg_map50_val':np.nan,
               'duration_sec':round(time.time()-t0,2)}
        if epoch % VAL_EVERY==0 or epoch==EPOCHS:
            metrics,_=coco_metric_for_model(model,val_ds,DEVICE,desc=f'Validation seed {seed}')
            entry['seg_map50_95_val']=metrics['seg_map50_95']
            entry['seg_map50_val']=metrics['seg_map50']
            ap=metrics['seg_map50_95']
            if ap>best_ap+0.0001:
                best_ap=ap;no_improve=0
                atomic_save({'model':cpu_model_state(model),'epoch':epoch,
                             'best_ap':best_ap,'yaml_sha256':YAML_SHA,'seed':seed},best)
                print(f'Seed {seed}: best validation mask AP={best_ap:.5f} at epoch {epoch}.')
            else:no_improve+=1
        rows.append(entry)
        pd.DataFrame(rows).to_csv(hist,index=False)
        atomic_save({'model':cpu_model_state(model),'optimizer':optimizer.state_dict(),
                     'scaler':scaler.state_dict(),'epoch':epoch,'best_ap':best_ap,
                     'no_improve':no_improve,'yaml_sha256':YAML_SHA,'seed':seed,
                     'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),
                     'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all()},ckpt)
        print(f'Seed={seed} epoch={epoch} loss={entry["train_loss_mean"]:.4f} valAP={entry["seg_map50_95_val"]} duration={entry["duration_sec"]}s')
        if no_improve>=PATIENCE_EVALS:
            print(f'Seed {seed}: early stopping after {PATIENCE_EVALS} consecutive non-improving validation checks.')
            break
    assert best.is_file() and hist.is_file(), f'Seed {seed}: missing outputs.'
    hist_df=pd.read_csv(hist)
    good=hist_df.dropna(subset=['seg_map50_95_val'])
    best_record=good.loc[good['seg_map50_95_val'].idxmax()]
    stored=torch.load(best,map_location='cpu',weights_only=False)
    assert abs(float(stored['best_ap'])-float(best_record['seg_map50_95_val']))<=0.00011, 'Best-checkpoint AP does not agree with validation history within the original 0.0001 improvement tolerance.'
    summary={'experiment':32,'seed':seed,'model':'MaskRCNN_ResNet50_FPN',
             'status':'completed','created_utc':datetime.now(timezone.utc).isoformat(),
             'best_epoch':int(stored['epoch']), 'best_validation_mask_AP':float(stored['best_ap']),
             'epochs_recorded':len(hist_df),'epochs_max':EPOCHS,
             'best_weights':str(best),'best_weights_sha256':sha256_file(best),
             'yaml_sha256':YAML_SHA,'gpu':torch.cuda.get_device_name(0),
             'torch':torch.__version__,'torchvision':torchvision.__version__,
             'batch':BATCH,'lr':LR,'weight_decay':WEIGHT_DECAY,
             'val_every':VAL_EVERY,'patience_evals':PATIENCE_EVALS,
             'test_evaluated':False}
    done.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
    del model,optimizer,scaler,loader,stored
    gc.collect();torch.cuda.empty_cache()
    return summary

new_results=[]
for sd in SEEDS:
    new_results.append(train_seed(sd))
print('Training complete:',[(d['seed'],round(d['best_validation_mask_AP'],5)) for d in new_results])


## 5. Generate multi-seed validation report (Excel, English figures, ZIP)

The baseline 2026 information is taken from its archived training record. It is **not rerun**. Models are compared in test later under the same COCOeval settings; no test AP is calculated here.


In [ ]:
orig_done=BASELINE/'ENTRENAMIENTO_FINALIZADO.json'
orig_hist=BASELINE/'historial_maskrcnn.csv'
assert orig_done.is_file() and orig_hist.is_file()
old=json.loads(orig_done.read_text(encoding='utf-8'))
assert int(old.get('seed'))==2026, 'Original seed mismatch.'
assert old.get('yaml_sha256')==YAML_SHA, 'Original Mask R-CNN used different frozen data YAML.'
oldbest=BASELINE/'maskrcnn_best_model.pt'
records=[{'seed':2026,'status':'original','best_epoch':int(old['best_epoch']),
          'best_validation_mask_AP':float(old['best_val_seg_map50_95']),
          'epochs_recorded':len(pd.read_csv(orig_hist)),
          'best_weights_sha256':sha256_file(oldbest),'best_weights':str(oldbest)}]
records.extend([{'seed':d['seed'],'status':'new', 'best_epoch':d['best_epoch'],
                 'best_validation_mask_AP':d['best_validation_mask_AP'],
                 'epochs_recorded':d['epochs_recorded'],
                 'best_weights_sha256':d['best_weights_sha256'],
                 'best_weights':d['best_weights']} for d in new_results])
summary=pd.DataFrame(records)
assert set(summary['seed'])=={2026,2027,2028}
full_history=[]
for seed in [2026,2027,2028]:
    path=orig_hist if seed==2026 else RUNS/f'seed_{seed}'/'historial_maskrcnn.csv'
    h=pd.read_csv(path);h.insert(0,'seed',seed);full_history.append(h)
history=pd.concat(full_history,ignore_index=True)
excel=REP/'CERDOS_32_MaskRCNN_MultiSeed_Validation.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as writer:
    summary.to_excel(writer,sheet_name='TrainingSummary',index=False)
    history.to_excel(writer,sheet_name='ValidationHistory',index=False)
plt.rcParams.update({'font.size':12,'axes.titlesize':14,'axes.labelsize':12,
                     'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})

def export_fig(fig,basename):
    result=[]
    for ext in ('png','pdf','svg'):
        target=FIG/f'{basename}.{ext}'
        fig.savefig(target,dpi=600 if ext=='png' else None,facecolor='white',bbox_inches='tight')
        result.append(target)
    plt.close(fig)
    return result
fig1,ax=plt.subplots(figsize=(9,5))
for sd,h in history.groupby('seed'):
    ax.plot(h['epoch'],h['seg_map50_95_val'],marker='o',linewidth=2,label=f'Seed {sd}')
ax.set(title='Mask R-CNN: Validation mask AP across training',xlabel='Epoch',ylabel='Mask AP50–95 (validation)')
ax.set_ylim(0,1);ax.set_xticks(list(range(2,EPOCHS+1,2)));ax.grid(axis='y',alpha=.2);ax.legend()
fig1.tight_layout()
p1=export_fig(fig1,'CERDOS_32_Validation_AP_Curves')
fig2,ax=plt.subplots(figsize=(8,5))
ax.bar(summary['seed'].astype(str),summary['best_validation_mask_AP'].to_numpy())
for i,v in enumerate(summary['best_validation_mask_AP']):ax.text(i,v+.012,f'{v:.4f}',ha='center')
ax.set(title='Mask R-CNN: Best validation mask AP by seed',xlabel='Training seed',ylabel='Best validation mask AP50–95',ylim=(0,1))
ax.grid(axis='y',alpha=.2)
fig2.tight_layout()
p2=export_fig(fig2,'CERDOS_32_Best_Validation_AP')
protocol={'experiment':32,'model':'MaskRCNN_ResNet50_FPN','original_seed':2026,
          'new_seeds':[2027,2028],
          'original_notebook':'06_CERDOS_Entrenamiento_MaskRCNN_GPU.ipynb',
          'epochs_max':EPOCHS,'batch':BATCH,'lr':LR,
          'weight_decay':WEIGHT_DECAY,'val_every':VAL_EVERY,'patience_evals':PATIENCE_EVALS,
          'pretrained':'torchvision COCO DEFAULT','optimizer':'AdamW','amp':True,
          'mask_min_size':512,'mask_max_size':768,
          'seed_file_hashes':EXPECTED_SHA,'device':torch.cuda.get_device_name(0),
          'torch':torch.__version__,'torchvision':torchvision.__version__,
          'python':platform.python_version(),
          'test_evaluated':False,
          'limitation':'Training budgets differ from YOLO11 and YOLO26. All metrics here are validation metrics.'}
pp=REP/'CERDOS_32_Protocol.json';pp.write_text(json.dumps(protocol,indent=2,ensure_ascii=False),encoding='utf-8')
report=REP/'CERDOS_32_Training_Report.json'
report.write_text(json.dumps({'experiment':32,'created_utc':datetime.now(timezone.utc).isoformat(),
                              'status':'completed','summary':records,'no_test_model_selection':True},
                             indent=2,ensure_ascii=False),encoding='utf-8')
zip_path=EXPORT/'CERDOS_32_MaskRCNN_Semillas_Resultados.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in [excel,pp,report,*p1,*p2]:z.write(f,arcname=f.name)
print('Results workbook:',excel)
print('ZIP:',zip_path,'MB',round(zip_path.stat().st_size/1e6,2))
print(summary[['seed','best_validation_mask_AP','best_epoch','epochs_recorded']].to_string(index=False))
from google.colab import files
files.download(str(zip_path))


## Interpretation and next step

- Mask R-CNN 2026/2027/2028 now have separate **validation training** records, selected checkpoints and hashes.
- Report differences between training budgets, dataset acquisition dates and small sample of three seeds; do not claim statistical equivalence or generalization to other farms.
- Next, harmonize **test inference and COCO mask evaluation** for the three seeds of each architecture with frozen settings. Do not adjust hyperparameters using test results.
- Send `CERDOS_32_MaskRCNN_Semillas_Resultados.zip` to review before proceeding. Full checkpoints stay in the Drive run folders.
